# MLFX Visual Analytics Notebook

This notebook focuses on visualization-first analysis of MLFX datasets, model artifacts, predictions, and performance metrics. It is designed to work directly against local project outputs and skip missing artifacts without breaking the rest of the notebook.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from mlfx.config.paths import ProjectPaths
from mlfx.evaluation.backtest import compute_metrics
from mlfx.evaluation.reporting import (
    get_drawdown_analysis_figure,
    get_equity_curve_figure,
    get_trade_distribution_figure,
    get_win_rate_figure,
)
from mlfx.training.data import load_labelled_dataset
from mlfx.training.feature_selection import select_numeric_feature_columns

px.defaults.template = "plotly_dark"
pd.options.display.max_columns = 120

SYMBOL = "XAUUSD"
TIMEFRAME = "1H"
LABEL = "label_10"
TOP_FEATURES = 12
DATA_TAIL = 600


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "config.toml").exists() and (candidate / "mlfx").exists():
            return candidate
    raise FileNotFoundError("Could not locate MLFX repository root.")


REPO_ROOT = find_repo_root()
PATHS = ProjectPaths(project_root=REPO_ROOT)


def read_json(path: Path) -> dict | list | None:
    if not path.exists():
        return None
    return json.loads(path.read_text())


def load_registry(paths: ProjectPaths) -> pd.DataFrame:
    registry_path = paths.models_root / "registry.json"
    data = read_json(registry_path)
    if not data:
        return pd.DataFrame()
    rows = []
    for entry in data:
        row = {
            "backend": entry.get("backend"),
            "symbol": entry.get("symbol"),
            "tf": entry.get("tf"),
            "label": entry.get("label"),
            "artifact_path": entry.get("artifact_path"),
            "registered_at": entry.get("registered_at"),
        }
        metrics = entry.get("metrics", {}) or {}
        for key, value in metrics.items():
            row[key] = value
        rows.append(row)
    return pd.DataFrame(rows)


def load_model_metrics(paths: ProjectPaths, symbol: str, tf: str, label: str) -> pd.DataFrame:
    metrics_dir = paths.models_label_dir(symbol, tf, label)
    rows = []
    if not metrics_dir.exists():
        return pd.DataFrame()
    for metrics_path in sorted(metrics_dir.glob("*.metrics.json")):
        payload = read_json(metrics_path)
        if not isinstance(payload, dict):
            continue
        row = {"metrics_file": metrics_path.name, "backend_guess": metrics_path.stem.replace(".metrics", "")}
        row.update(payload)
        rows.append(row)
    return pd.DataFrame(rows)


def latest_prediction_path(paths: ProjectPaths, symbol: str, tf: str, label: str) -> Path | None:
    path = paths.predictions_label_dir(symbol, tf, label) / "predictions.parquet"
    return path if path.exists() else None


def latest_trade_report_path(paths: ProjectPaths, symbol: str, tf: str, label: str) -> Path | None:
    report_root = paths.reports_label_dir(symbol, tf, label)
    if not report_root.exists():
        return None
    candidates = sorted(report_root.rglob("*_trades.parquet"), key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None


def metrics_log_path(paths: ProjectPaths, symbol: str, tf: str, label: str) -> Path:
    return paths.runs_label_dir(symbol, tf, label) / "metrics_log.jsonl"


print(REPO_ROOT)


## 1. Dataset Snapshot

Load the local labelled dataset and summarize its size, timeline, and dominant engineered features.

In [ ]:
dataset = load_labelled_dataset(SYMBOL, TIMEFRAME, paths=PATHS)
if dataset is None or dataset.is_empty():
    raise RuntimeError(f"No labelled dataset found for {SYMBOL}/{TIMEFRAME}.")

feature_cols = select_numeric_feature_columns(dataset)
dataset_summary = pd.DataFrame({
    "rows": [dataset.height],
    "columns": [dataset.width],
    "feature_columns": [len(feature_cols)],
    "start": [dataset['timestamp'].min() if 'timestamp' in dataset.columns else None],
    "end": [dataset['timestamp'].max() if 'timestamp' in dataset.columns else None],
})
display(dataset_summary)
display(dataset.head(5).to_pandas())


In [ ]:
tail_df = dataset.tail(min(DATA_TAIL, dataset.height)).to_pandas()
if {"timestamp", "close"}.issubset(tail_df.columns):
    fig = px.line(tail_df, x="timestamp", y="close", title=f"Close Price Timeline: {SYMBOL} {TIMEFRAME}")
    fig.show()

available_features = [c for c in ["rsi_14", "ema_20", "ema_50", "ema_200", "atr_14", "macd"] if c in tail_df.columns]
if available_features:
    fig = px.line(tail_df, x="timestamp", y=available_features, title="Indicator Timeline")
    fig.show()


## 2. Feature Visuals

Look at label balance, feature correlations, and a compact feature-distribution panel.

In [ ]:
if LABEL in dataset.columns:
    label_balance = dataset.group_by(LABEL).len().sort(LABEL).rename({"len": "count"}).to_pandas()
    fig = px.bar(label_balance, x=LABEL, y="count", title=f"Label Balance: {LABEL}")
    fig.show()

focus_features = feature_cols[:TOP_FEATURES]
if len(focus_features) >= 2:
    corr = dataset.select(focus_features).to_pandas().corr(numeric_only=True)
    fig = px.imshow(corr, color_continuous_scale="RdBu", zmin=-1, zmax=1, title="Feature Correlation Matrix")
    fig.show()

if focus_features:
    melted = dataset.select(focus_features[:6]).to_pandas().melt(var_name="feature", value_name="value")
    fig = px.box(melted, x="feature", y="value", points=False, title="Feature Distribution Snapshot")
    fig.show()


## 3. Model Artifact Metrics

Compare backend metrics from saved model metric files and the registry.

In [ ]:
metrics_df = load_model_metrics(PATHS, SYMBOL, TIMEFRAME, LABEL)
registry_df = load_registry(PATHS)
registry_slice = registry_df[(registry_df.get('symbol') == SYMBOL) & (registry_df.get('tf') == TIMEFRAME) & (registry_df.get('label') == LABEL)] if not registry_df.empty else pd.DataFrame()

display(metrics_df)
display(registry_slice)


In [ ]:
if not metrics_df.empty:
    metric_candidates = [c for c in ["best_cv_f1_macro", "f1_macro_train", "f1_macro_oos", "n_samples"] if c in metrics_df.columns]
    if metric_candidates:
        plot_df = metrics_df[["metrics_file", *metric_candidates]].copy()
        plot_df["model"] = plot_df["metrics_file"].str.replace(".metrics.json", "", regex=False)
        melted = plot_df.melt(id_vars=["model"], value_vars=metric_candidates, var_name="metric", value_name="value")
        fig = px.bar(melted, x="model", y="value", color="metric", barmode="group", title="Saved Model Metrics")
        fig.show()

if not registry_slice.empty and "best_cv_f1_macro" in registry_slice.columns:
    registry_plot = registry_slice.copy()
    registry_plot["best_cv_f1_macro"] = pd.to_numeric(registry_plot["best_cv_f1_macro"], errors="coerce")
    registry_plot = registry_plot.dropna(subset=["best_cv_f1_macro"])
    if not registry_plot.empty:
        fig = px.scatter(
            registry_plot,
            x="backend",
            y="best_cv_f1_macro",
            size="best_cv_f1_macro",
            color="backend",
            title="Registry Best CV F1 by Backend",
        )
        fig.show()
    else:
        print("Registry entries exist, but none contain a numeric best_cv_f1_macro value yet.")


## 4. Predictions Visuals

Render prediction distributions and price overlays when batch prediction artifacts are available.

In [ ]:
prediction_path = latest_prediction_path(PATHS, SYMBOL, TIMEFRAME, LABEL)
predictions_df = pl.read_parquet(prediction_path) if prediction_path else None
print(f"Prediction artifact: {prediction_path}")

if predictions_df is None:
    print("Predictions artifact not found. Run batch inference to populate this section.")
else:
    display(predictions_df.head(5).to_pandas())

    pred_pd = predictions_df.to_pandas()
    if "prediction" in pred_pd.columns:
        fig = px.histogram(pred_pd, x="prediction", nbins=10, title="Prediction Distribution")
        fig.show()

    if LABEL in pred_pd.columns and "prediction" in pred_pd.columns:
        compare = pd.crosstab(pred_pd[LABEL], pred_pd["prediction"])
        display(compare)

    if {"timestamp", "close", "prediction"}.issubset(pred_pd.columns):
        pred_tail = pred_pd.tail(min(DATA_TAIL, len(pred_pd)))
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=pred_tail["timestamp"], y=pred_tail["close"], mode="lines", name="close"))
        fig.add_trace(go.Scatter(x=pred_tail["timestamp"], y=pred_tail["close"], mode="markers", name="prediction", marker=dict(color=pred_tail["prediction"], colorscale="Viridis", size=7, showscale=True)))
        fig.update_layout(title="Prediction Overlay on Price")
        fig.show()


## 5. Backtest Metrics and Performance

Load the latest trade artifact and compute a compact performance card set plus visual backtest analytics.

In [ ]:
trade_path = latest_trade_report_path(PATHS, SYMBOL, TIMEFRAME, LABEL)
trades_df = pl.read_parquet(trade_path) if trade_path else None
print(f"Trades artifact: {trade_path}")

if trades_df is None or trades_df.is_empty():
    print("Trade report not found. Run evaluation to populate this section.")
else:
    metrics = compute_metrics(trades_df, initial_capital=10_000.0, risk_pct=1.0)
    metrics_card = pd.DataFrame([metrics])[[
        "total_trades",
        "win_rate",
        "total_r",
        "max_drawdown_r",
        "profit_factor",
        "sharpe_ratio",
        "sortino_ratio",
        "calmar_ratio",
        "final_capital",
    ]]
    display(metrics_card)

    get_equity_curve_figure(trades_df, f"Equity Curve: {SYMBOL} {TIMEFRAME} ({LABEL})").show()
    get_drawdown_analysis_figure(trades_df, f"Drawdown Analysis: {SYMBOL} {TIMEFRAME} ({LABEL})").show()
    get_trade_distribution_figure(trades_df, f"Trade Distribution: {SYMBOL} {TIMEFRAME} ({LABEL})").show()
    get_win_rate_figure(trades_df, f"Win Rate Analysis: {SYMBOL} {TIMEFRAME} ({LABEL})").show()


## 6. Training Run Metrics History

Plot the historical metrics log so you can spot drift in model quality or runtime cost across runs.

In [ ]:
run_log = metrics_log_path(PATHS, SYMBOL, TIMEFRAME, LABEL)
print(f"Metrics log path: {run_log}")

if not run_log.exists():
    print("metrics_log.jsonl not found for this label.")
else:
    records = [json.loads(line) for line in run_log.read_text().splitlines() if line.strip()]
    history_df = pd.DataFrame(records)
    display(history_df.tail(10))

    plot_metrics = [c for c in ["best_cv_f1_macro", "accuracy", "elapsed_seconds"] if c in history_df.columns]
    if plot_metrics:
        fig = px.line(history_df.reset_index(drop=True), x=history_df.index, y=plot_metrics, title="Run History Metrics")
        fig.update_layout(xaxis_title="run index")
        fig.show()
